# LeetCode 338 — Counting Bits bằng Python

**Link đề:** https://leetcode.com/problems/counting-bits/
**Độ khó:** Easy · **Chủ đề:** Dynamic Programming, Bit Manipulation

```python
class Solution:
    def countBits(self, n: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Đọc được **biểu diễn nhị phân** và hiểu "số bit 1" (popcount) của một số.
2. Nắm các toán tử bit `&`, `>>`, và mẹo `i & (i - 1)`.
3. Đi từ cách **đếm riêng từng số** `O(n log n)` tới **quy hoạch động (DP)** `O(n)` — dùng lại kết quả đã tính, đúng như gợi ý của đề.
4. Hiểu 3 công thức DP: theo **dịch phải** `ans[i >> 1]`, theo **bit thấp nhất** `ans[i & (i-1)]`, theo **bit cao nhất** (offset lũy thừa 2).
5. Trace bảng DP, chứng minh từng công thức, test đối chiếu và tự giải biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho số nguyên `n`. Trả về mảng `ans` dài `n + 1`, trong đó với mỗi `i` (`0 <= i <= n`), `ans[i]` là **số chữ số 1** trong biểu diễn nhị phân của `i`.

| Input | Output | Giải thích |
|---|---|---|
| `n = 2` | `[0, 1, 1]` | `0 → 0`, `1 → 1`, `2 → 10` |
| `n = 5` | `[0, 1, 1, 2, 1, 2]` | `3 → 11`, `4 → 100`, `5 → 101` |

**Constraints:** `0 <= n <= 10^5`.

**Follow up:**
- Dễ nghĩ ra lời giải `O(n log n)`. Bạn có làm được trong **`O(n)`**, một lượt duyệt không?
- Đừng dùng hàm dựng sẵn kiểu `__builtin_popcount` (C++) — trong Python tương đương `bin(i).count("1")` hay `int.bit_count()`.

**Gợi ý của đề:** Dùng lại những gì đã tính. Chia số theo các khoảng `[2-3]`, `[4-7]`, `[8-15]`… Hoặc: tính chẵn/lẻ của số có giúp gì không?

In [1]:
for i in range(6):
    print(i, "->", format(i, "b"))   # format(i, "b") cho chuỗi nhị phân không có tiền tố 0b

0 -> 0
1 -> 1
2 -> 10
3 -> 11
4 -> 100
5 -> 101


## 2. Nền tảng: nhị phân và toán tử bit

### 2.1. Hệ nhị phân
Mỗi vị trí mạnh gấp 2 lần vị trí bên phải. `13 = 8 + 4 + 1 = 1101₂`.

| Vị trí | 3 | 2 | 1 | 0 |
|---|---|---|---|---|
| Giá trị vị trí | 8 | 4 | 2 | 1 |
| Bit của 13 | 1 | 1 | 0 | 1 |

`bin(13)` trả chuỗi `'0b1101'` (tiền tố `0b` báo hiệu nhị phân).

In [2]:
print(bin(13))            # '0b1101'
print(bin(13)[2:])        # '1101' — cắt bỏ '0b'
print(int("1101", 2))     # 13 — đọc ngược từ chuỗi nhị phân
print(bin(13).count("1")) # 3 — popcount bằng chuỗi (cách "dựng sẵn")

0b1101
1101
13
3


### 2.2. `&` (AND từng bit) và `i & 1` — lấy bit cuối

`a & b` so từng cặp bit: kết quả 1 chỉ khi **cả hai** là 1. Vì `1 = ...0001`, nên `i & 1` giữ lại đúng **bit cuối** của `i`: `1` nếu `i` lẻ, `0` nếu chẵn.

In [3]:
for i in [4, 5, 6, 7]:
    print(f"{i} = {i:04b} | i & 1 = {i & 1} | i % 2 = {i % 2}")

4 = 0100 | i & 1 = 0 | i % 2 = 0
5 = 0101 | i & 1 = 1 | i % 2 = 1
6 = 0110 | i & 1 = 0 | i % 2 = 0
7 = 0111 | i & 1 = 1 | i % 2 = 1


### 2.3. `>>` (dịch phải) — bỏ bit cuối

`i >> 1` dịch mọi bit sang phải một vị trí, bit cuối rơi mất. Tương đương `i // 2`.

`13 = 1101₂` → `13 >> 1 = 110₂ = 6`.

In [4]:
for i in [13, 6, 7, 1]:
    print(f"{i:>2} = {i:04b} -> i >> 1 = {i >> 1:>2} = {i >> 1:04b} | i // 2 = {i // 2}")

13 = 1101 -> i >> 1 =  6 = 0110 | i // 2 = 6
 6 = 0110 -> i >> 1 =  3 = 0011 | i // 2 = 3
 7 = 0111 -> i >> 1 =  3 = 0011 | i // 2 = 3
 1 = 0001 -> i >> 1 =  0 = 0000 | i // 2 = 0


### 2.4. `i & (i - 1)` — xóa bit 1 thấp nhất

Trừ 1 làm bit 1 thấp nhất thành 0 và mọi bit 0 phía sau nó thành 1. AND với số gốc → xóa đúng bit 1 thấp nhất.

`12 = 1100₂`, `11 = 1011₂` → `12 & 11 = 1000₂ = 8`.

In [5]:
for i in [12, 7, 8, 10]:
    print(f"i = {i:>2} = {i:04b} | i-1 = {i - 1:04b} | i & (i-1) = {i & (i - 1):04b} = {i & (i - 1)}")

i = 12 = 1100 | i-1 = 1011 | i & (i-1) = 1000 = 8
i =  7 = 0111 | i-1 = 0110 | i & (i-1) = 0110 = 6
i =  8 = 1000 | i-1 = 0111 | i & (i-1) = 0000 = 0
i = 10 = 1010 | i-1 = 1001 | i & (i-1) = 1000 = 8


## 3. Cách trực giác — đếm riêng từng số

Với mỗi `i`, đếm bit 1 bằng cách lặp: lấy bit cuối `i & 1`, cộng vào, rồi `i >>= 1` cho tới khi `i == 0`.

Đây là cách "không dùng built-in". Mỗi số `i` có khoảng `log₂ i` bit → tổng `O(n log n)`.

In [6]:
def popcount_loop(x: int) -> int:
    count = 0
    while x > 0:
        count += x & 1
        x >>= 1
    return count

def count_bits_naive(n: int) -> list[int]:
    return [popcount_loop(i) for i in range(n + 1)]

print(count_bits_naive(2))
print(count_bits_naive(5))

[0, 1, 1]
[0, 1, 1, 2, 1, 2]


### 3.1. Biến thể của cách trực giác — Brian Kernighan

Thay vì đi qua **mọi** bit, chỉ lặp **đúng số bit 1** bằng `x &= x - 1`. Vẫn `O(n log n)` worst case.

In [7]:
def popcount_kernighan(x: int) -> int:
    count = 0
    while x:
        x &= x - 1      # xóa bit 1 thấp nhất
        count += 1
    return count

print([popcount_kernighan(i) for i in range(9)])

[0, 1, 1, 2, 1, 2, 2, 3, 1]


## 4. Nút thắt — tính lại những gì đã biết

Khi tính popcount của `13 = 1101₂` bằng vòng lặp, ta lần lượt đi qua `13 → 6 → 3 → 1 → 0`. Nhưng `6`, `3`, `1` **đã được tính trước đó** (vì ta đi từ 0 lên `n`)! Ta đang lặp lại công việc.

Đếm tổng số vòng lặp của cách trực giác để thấy chi phí `n log n`:

In [8]:
def total_loop_steps(n: int) -> int:
    steps = 0
    for i in range(n + 1):
        x = i
        while x > 0:
            x >>= 1
            steps += 1
    return steps

for n in [10, 1000, 100000]:
    print(f"n = {n:>6}: tổng vòng lặp = {total_loop_steps(n):>8} | n + 1 = {n + 1}")

n =     10: tổng vòng lặp =       29 | n + 1 = 11
n =   1000: tổng vòng lặp =     8987 | n + 1 = 1001
n = 100000: tổng vòng lặp =  1568946 | n + 1 = 100001


## 5. Insight — số bit 1 của `i` = số bit 1 của `i >> 1` + bit cuối của `i`

`i >> 1` là `i` bỏ đi bit cuối. Vậy:

```text
popcount(i) = popcount(i >> 1) + (i & 1)
```

Ví dụ `13 = 1101₂`: `13 >> 1 = 6 = 110₂` có 2 bit 1; bit cuối của 13 là 1 → `2 + 1 = 3` ✔.

Vì `i >> 1 < i` (với `i >= 1`), khi tính `ans[i]` thì `ans[i >> 1]` **đã có sẵn** → mỗi ô tính trong `O(1)`. Đây là **DP**: trạng thái `ans[i]`, công thức chuyển `ans[i] = ans[i >> 1] + (i & 1)`, khởi tạo `ans[0] = 0`.

Gợi ý "chẵn/lẻ" của đề chính là ý này: số chẵn `2k` có cùng số bit 1 với `k`; số lẻ `2k + 1` có thêm 1.

In [9]:
i = 13
print(f"{i} = {i:b}, {i} >> 1 = {i >> 1} = {i >> 1:b}")
print("popcount(6) =", popcount_loop(6), "| bit cuối của 13 =", i & 1, "| tổng =", popcount_loop(6) + (i & 1))
print("popcount(13) =", popcount_loop(13))

13 = 1101, 13 >> 1 = 6 = 110
popcount(6) = 2 | bit cuối của 13 = 1 | tổng = 3
popcount(13) = 3


## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — tạo mảng kết quả dài `n + 1`, `ans[0] = 0`:**
```python
ans = [0] * (n + 1)
```

**Đoạn 2 — duyệt `i` từ 1 tới `n` (thứ tự tăng dần quan trọng!):**
```python
for i in range(1, n + 1):
```

**Đoạn 3 — công thức DP:**
```python
ans[i] = ans[i >> 1] + (i & 1)
```
Dấu ngoặc `(i & 1)` cần thiết vì trong Python `+` ưu tiên **cao hơn** `&`.

**Đoạn 4 — trả `ans`.** Với `n = 0` vòng lặp không chạy, trả `[0]`.

In [10]:
print(2 + 3 & 1)      # (2 + 3) & 1 = 5 & 1 = 1  — + được tính trước
print(2 + (3 & 1))    # 2 + 1 = 3

def count_bits_dp_shift(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans

print(count_bits_dp_shift(0))
print(count_bits_dp_shift(5))
print(count_bits_dp_shift(16))

1
3
[0]
[0, 1, 1, 2, 1, 2]
[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1]


## 7. TRACE từng bước

Bảng tay cho `n = 8`:

| i | nhị phân | i >> 1 | ans[i >> 1] | i & 1 | ans[i] |
|---:|---:|---:|---:|---:|---:|
| 1 | 1 | 0 | 0 | 1 | 1 |
| 2 | 10 | 1 | 1 | 0 | 1 |
| 3 | 11 | 1 | 1 | 1 | 2 |
| 4 | 100 | 2 | 1 | 0 | 1 |
| 5 | 101 | 2 | 1 | 1 | 2 |
| 6 | 110 | 3 | 2 | 0 | 2 |
| 7 | 111 | 3 | 2 | 1 | 3 |
| 8 | 1000 | 4 | 1 | 0 | 1 |

Ý nghĩa các cột: `i >> 1` là ô "cha" đã tính; `i & 1` là phần bổ sung; `ans[i]` là tổng hai cột đó.

In [11]:
def trace_count_bits(n: int) -> list[int]:
    ans = [0] * (n + 1)
    print(" i | bin  | i>>1 | ans[i>>1] | i&1 | ans[i]")
    for i in range(1, n + 1):
        parent = i >> 1
        ans[i] = ans[parent] + (i & 1)
        print(f"{i:>2} | {i:>4b} | {parent:>4} | {ans[parent]:>9} | {i & 1:>3} | {ans[i]:>6}")
    print("ans =", ans)
    return ans

trace_count_bits(8)

 i | bin  | i>>1 | ans[i>>1] | i&1 | ans[i]
 1 |    1 |    0 |         0 |   1 |      1
 2 |   10 |    1 |         1 |   0 |      1
 3 |   11 |    1 |         1 |   1 |      2
 4 |  100 |    2 |         1 |   0 |      1
 5 |  101 |    2 |         1 |   1 |      2
 6 |  110 |    3 |         2 |   0 |      2
 7 |  111 |    3 |         2 |   1 |      3
 8 | 1000 |    4 |         1 |   0 |      1
ans = [0, 1, 1, 2, 1, 2, 2, 3, 1]


[0, 1, 1, 2, 1, 2, 2, 3, 1]

## 8. Hai công thức DP khác (theo gợi ý của đề)

### Cách DP 2 — xóa bit 1 thấp nhất: `ans[i] = ans[i & (i - 1)] + 1`

`i & (i - 1)` nhỏ hơn `i` và có **ít hơn đúng 1** bit 1. Ví dụ `12 = 1100₂` → `8 = 1000₂`: `ans[12] = ans[8] + 1 = 2`.

In [12]:
def count_bits_dp_lowbit(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i & (i - 1)] + 1
    return ans

print(count_bits_dp_lowbit(16))

[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1]


### Cách DP 3 — theo khoảng `[2^k, 2^(k+1) - 1]` (bit cao nhất)

Gợi ý "chia theo khoảng `[2-3]`, `[4-7]`, `[8-15]`": mỗi số trong khoảng `[2^k, 2^(k+1))` bằng `2^k + j` với `0 <= j < 2^k`. Bit cao nhất đóng góp 1, phần còn lại là `j`:

```text
ans[2^k + j] = ans[j] + 1
```

Nói cách khác, khoảng `[4..7]` = khoảng `[0..3]` cộng 1: `[0,1,1,2] → [1,2,2,3]`.

In [13]:
def count_bits_dp_highbit(n: int) -> list[int]:
    ans = [0] * (n + 1)
    offset = 1                    # lũy thừa của 2 lớn nhất <= i
    for i in range(1, n + 1):
        if offset * 2 == i:       # sang khoảng mới
            offset = i
        ans[i] = ans[i - offset] + 1
    return ans

print(count_bits_dp_highbit(16))

# Minh họa "khoảng mới = khoảng cũ + 1"
ans = count_bits_dp_highbit(15)
for k in range(4):
    lo, hi = 2 ** k, 2 ** (k + 1) - 1
    print(f"[{lo}..{hi}]: {ans[lo:hi + 1]}  <-  [0..{hi - lo}] + 1 = {[x + 1 for x in ans[0:hi - lo + 1]]}")

[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1]
[1..1]: [1]  <-  [0..0] + 1 = [1]
[2..3]: [1, 2]  <-  [0..1] + 1 = [1, 2]
[4..7]: [1, 2, 2, 3]  <-  [0..3] + 1 = [1, 2, 2, 3]
[8..15]: [1, 2, 2, 3, 2, 3, 3, 4]  <-  [0..7] + 1 = [1, 2, 2, 3, 2, 3, 3, 4]


## 9. Code nộp LeetCode

### Cách chính — DP dịch phải, `O(n)`

In [14]:
class Solution:
    def countBits(self, n: int) -> list[int]:
        ans = [0] * (n + 1)
        for i in range(1, n + 1):
            ans[i] = ans[i >> 1] + (i & 1)
        return ans

print(Solution().countBits(2), Solution().countBits(5))

[0, 1, 1] [0, 1, 1, 2, 1, 2]


### Cách nộp thay thế — DP xóa bit thấp nhất

In [15]:
class SolutionLowbit:
    def countBits(self, n: int) -> list[int]:
        ans = [0] * (n + 1)
        for i in range(1, n + 1):
            ans[i] = ans[i & (i - 1)] + 1
        return ans

print(SolutionLowbit().countBits(5))

[0, 1, 1, 2, 1, 2]


### Cách nộp đơn giản (không theo follow-up) — `bin(i).count("1")`
Được chấp nhận nhưng dùng "built-in" và `O(n log n)`. Python 3.10+ còn có `i.bit_count()`.

In [16]:
class SolutionBuiltin:
    def countBits(self, n: int) -> list[int]:
        return [bin(i).count("1") for i in range(n + 1)]

print(SolutionBuiltin().countBits(5))

[0, 1, 1, 2, 1, 2]


## 10. Vì sao đúng?

**Công thức dịch phải.** Viết `i = 2·q + r` với `q = i >> 1`, `r = i & 1 ∈ {0, 1}`. Nhị phân của `i` là nhị phân của `q` nối thêm bit `r` ở cuối. Nên `popcount(i) = popcount(q) + r`.

**Thứ tự tính (invariant của vòng lặp):** trước khi tính `ans[i]`, mọi `ans[j]` với `j < i` đã đúng. Vì `q = i >> 1 < i` khi `i >= 1`, ta dùng được `ans[q]`. Quy nạp từ `ans[0] = 0` → mọi ô đều đúng.

**Công thức bit thấp nhất:** `i & (i - 1)` bằng `i` bỏ đúng một bit 1 → ít hơn 1 bit 1 và nhỏ hơn `i`.

**Công thức bit cao nhất:** với `2^k <= i < 2^(k+1)`, `i - 2^k` là `i` bỏ bit cao nhất (bit thứ `k`) → ít hơn 1 bit 1.

## 11. Time / Space Complexity và so sánh

Gọi `n` là input; số bit của `n` là `log₂ n + 1`.

| Cách | Time | Space phụ (ngoài mảng kết quả) | Theo follow-up? |
|---|---|---|:---:|
| Vòng lặp từng bit | `O(n log n)` | `O(1)` | không |
| Kernighan từng số | `O(n log n)` worst | `O(1)` | không |
| `bin(i).count("1")` | `O(n log n)` | `O(log n)` chuỗi tạm | không (built-in) |
| **DP `i >> 1`** | **`O(n)`** | **`O(1)`** | **có** |
| DP `i & (i-1)` | `O(n)` | `O(1)` | có |
| DP offset bit cao | `O(n)` | `O(1)` | có |

Mảng kết quả `ans` luôn tốn `O(n)` vì đề yêu cầu trả về.

In [17]:
import time
n = 100000
for name, f in [("naive loop", count_bits_naive), ("DP shift", count_bits_dp_shift),
                ("DP lowbit", count_bits_dp_lowbit), ("DP highbit", count_bits_dp_highbit)]:
    t0 = time.perf_counter()
    res = f(n)
    print(f"{name:<11} sum = {sum(res)} | ≈ {time.perf_counter() - t0:.3f}s")

naive loop  sum = 815030 | ≈ 0.090s
DP shift    sum = 815030 | ≈ 0.006s
DP lowbit   sum = 815030 | ≈ 0.006s
DP highbit  sum = 815030 | ≈ 0.007s


## 12. Bộ test — mẫu đề, biên, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual if len(actual) <= 20 else str(actual[:10]) + '...'}")

sol = Solution()
check("đề: n=2", sol.countBits(2), [0, 1, 1])
check("đề: n=5", sol.countBits(5), [0, 1, 1, 2, 1, 2])
check("biên: n=0", sol.countBits(0), [0])
check("n=1", sol.countBits(1), [0, 1])
check("lũy thừa 2: n=16", sol.countBits(16), [0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1])
big = sol.countBits(10**5)
assert len(big) == 10**5 + 1
assert big[65535] == 16 and big[65536] == 1 and big[10**5] == bin(10**5).count("1")
print("PASS n = 10^5: độ dài, 2^16 - 1 có 16 bit, 2^16 có 1 bit")

PASS đề: n=2: [0, 1, 1]
PASS đề: n=5: [0, 1, 1, 2, 1, 2]
PASS biên: n=0: [0]
PASS n=1: [0, 1]
PASS lũy thừa 2: n=16: [0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1]
PASS n = 10^5: độ dài, 2^16 - 1 có 16 bit, 2^16 có 1 bit


In [19]:
reference = [bin(i).count("1") for i in range(5001)]
methods = [count_bits_naive, count_bits_dp_shift, count_bits_dp_lowbit, count_bits_dp_highbit,
           Solution().countBits, SolutionLowbit().countBits, SolutionBuiltin().countBits,
           lambda n: [popcount_kernighan(i) for i in range(n + 1)]]
for n in [0, 1, 2, 3, 7, 8, 100, 1023, 1024, 5000]:
    for f in methods:
        assert f(n) == reference[:n + 1], (f, n)
print("8 cách khớp với tham chiếu trên nhiều n (gồm 0, lũy thừa 2 và 2^k - 1)")

8 cách khớp với tham chiếu trên nhiều n (gồm 0, lũy thừa 2 và 2^k - 1)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Thiếu ngoặc `ans[i >> 1] + i & 1`
Python tính `+` trước `&` → `(ans[i >> 1] + i) & 1` → chỉ ra 0 hoặc 1.

In [20]:
def count_bits_precedence_bug(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + i & 1        # SAI: thiếu ngoặc
    return ans

print("sai :", count_bits_precedence_bug(8))
print("đúng:", count_bits_dp_shift(8))

sai : [0, 1, 1, 0, 1, 0, 0, 1, 1]
đúng: [0, 1, 1, 2, 1, 2, 2, 3, 1]


### Lỗi 2: Mảng dài `n` thay vì `n + 1`
Đề cần cả `ans[n]` → mảng phải dài `n + 1`; nếu không sẽ `IndexError` hoặc thiếu phần tử cuối.

### Lỗi 3: Dùng `i << 1` (dịch trái) thay vì `i >> 1`
`i << 1 = 2i` lớn hơn `i` → chưa được tính, hoặc vượt chỉ số.

In [21]:
def count_bits_short_array(n: int) -> list[int]:
    ans = [0] * n                           # SAI: thiếu 1 ô
    for i in range(1, n):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans

print("sai :", count_bits_short_array(5), "| đúng:", count_bits_dp_shift(5))
try:
    ans = [0] * 6
    for i in range(1, 6):
        ans[i] = ans[i << 1] + (i & 1)      # SAI: i << 1 = 2i
except IndexError as e:
    print("IndexError với i << 1:", e)

sai : [0, 1, 1, 2, 1] | đúng: [0, 1, 1, 2, 1, 2]
IndexError với i << 1: list index out of range


### Lỗi 4: Mang thói quen độ ưu tiên toán tử từ C/Java sang (hoặc ngược lại)
Trong **C/Java**, `i & (i - 1) == 0` bị hiểu là `i & ((i - 1) == 0)` vì `==` ưu tiên cao hơn `&` → bug kinh điển khi kiểm tra lũy thừa của 2 (dùng trong DP bit cao).
Trong **Python** thì ngược lại: `&` ưu tiên **cao hơn** `==`, nên biểu thức trên chạy đúng. Nhưng `+` lại cao hơn `&` (Lỗi 1). Cách an toàn ở mọi ngôn ngữ: **luôn viết ngoặc** quanh biểu thức bit.

In [22]:
i = 8
print(i & (i - 1) == 0)       # Python hiểu là (8 & 7) == 0 -> True
print((i & (i - 1)) == 0)     # True — viết ngoặc rõ ràng, đúng ở mọi ngôn ngữ
print(i & ((i - 1) == 0))     # cách C/Java hiểu khi thiếu ngoặc: 8 & False = 0 -> sai ý đồ

True
True
0


## 14. Mẹo nhận dạng pattern

- **"Trả về kết quả cho mọi `i` từ 0 tới n"** + mỗi kết quả liên quan tới số nhỏ hơn → **DP trên chỉ số**.
- Bài bit: tìm quan hệ giữa `i` và một số **nhỏ hơn** sinh ra từ `i` bằng phép bit: `i >> 1`, `i & (i - 1)`, `i - highbit`.
- Follow-up "O(n), một lượt" → mỗi ô phải tính `O(1)` từ ô đã có.
- Độ ưu tiên trong Python (cao → thấp): `+ -` > `<< >>` > `&` > `^` > `|` > so sánh (`==`, `<`, …). Kiểm chứng ở cell dưới — và vẫn **luôn dùng ngoặc**.

In [23]:
print(1 + 2 << 1)        # (1 + 2) << 1 = 6      : + cao hơn <<
print(2 + 3 & 1)         # (2 + 3) & 1 = 1       : + cao hơn &
print(6 & 3 == 2)        # (6 & 3) == 2 -> True  : & cao hơn ==

6
1
True


## 15. Biến thể

### Biến thể 1 — Number of 1 Bits (LeetCode 191) cho một số
Dùng `popcount_kernighan(x)`.

### Biến thể 2 — tổng số bit 1 của mọi số từ 0 tới n
Chỉ cần `sum(countBits(n))`, hoặc tính trực tiếp theo từng vị trí bit trong `O(log n)`.

### Biến thể 3 — đếm số trong `[0, n]` có số bit 1 là số nguyên tố (gần LeetCode 762)

In [24]:
def total_bits_up_to(n: int) -> int:
    # Đếm theo từng vị trí bit k: bit k lặp theo chu kỳ 2^(k+1): 2^k số 0 rồi 2^k số 1
    total = 0
    k = 0
    while (1 << k) <= n:
        cycle = 1 << (k + 1)
        full = (n + 1) // cycle
        total += full * (1 << k) + max(0, (n + 1) % cycle - (1 << k))
        k += 1
    return total

def count_prime_set_bits(n: int) -> int:
    primes = {2, 3, 5, 7, 11, 13, 17, 19}
    return sum(1 for c in count_bits_dp_shift(n) if c in primes)

print(total_bits_up_to(5), sum(count_bits_dp_shift(5)))          # 7 7
print(total_bits_up_to(1000), sum(count_bits_dp_shift(1000)))
print(count_prime_set_bits(10))

7 7
4938 4938
6


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Lập bảng DP `ans[i] = ans[i >> 1] + (i & 1)` cho `i = 9..15` (dùng kết quả `ans[0..8]` ở mục 7).

### Bài 2 — Dự đoán output
Đoán: `13 >> 1`, `13 & 1`, `12 & 11`, `bin(10)[2:]`, `3 + 4 & 1`.

### Bài 3 — Điền chỗ trống

In [25]:
def exercise_count_bits(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        # TODO: công thức DP dùng i >> 1 và i & 1
        pass
    return ans

# assert exercise_count_bits(5) == [0, 1, 1, 2, 1, 2]
# assert exercise_count_bits(0) == [0]

### Bài 4 — Tự viết: DP theo `i & (i - 1)` không nhìn lại mục 8

In [26]:
def exercise_count_bits_lowbit(n: int) -> list[int]:
    # TODO
    pass

# assert exercise_count_bits_lowbit(8) == [0, 1, 1, 2, 1, 2, 2, 3, 1]

### Bài 5 — Biến thể: đếm số bit 0 (trong biểu diễn không có số 0 vô nghĩa ở đầu)
Trả về mảng `zeros` với `zeros[i]` = số bit 0 của `i` (quy ước `zeros[0] = 1` vì `0` viết là `"0"`). Hãy làm bằng DP `O(n)`.

In [27]:
def exercise_count_zero_bits(n: int) -> list[int]:
    # TODO: với i >= 2: zeros[i] = zeros[i >> 1] + (1 - (i & 1)); xử lý riêng i = 0, 1
    pass

# assert exercise_count_zero_bits(8) == [1, 0, 1, 0, 2, 1, 1, 0, 3]

### Bài 6 — Mini challenge: sắp xếp theo số bit 1 (LeetCode 1356)
Sắp xếp `arr` tăng dần theo số bit 1; nếu bằng nhau thì theo giá trị. Dùng `sorted(..., key=...)` với key là **tuple**.

In [28]:
def exercise_sort_by_bits(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_sort_by_bits([0, 1, 2, 3, 4, 5, 6, 7, 8]) == [0, 1, 2, 4, 8, 3, 5, 6, 7]
# assert exercise_sort_by_bits([1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]) == [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

## 17. Gợi ý 3 mức

**Bài 1** — (1) `9 >> 1 = 4`. (2) Dùng `ans[4] = 1`, `9 & 1 = 1`. (3) Kết quả mong đợi `[2, 2, 3, 2, 3, 3, 4]`.

**Bài 2** — (1) Dịch phải = chia 2 lấy nguyên. (2) `12 = 1100`, `11 = 1011`. (3) Nhớ `+` trước `&`.

**Bài 3** — (1) Ô "cha" là `i >> 1`. (2) Cộng bit cuối. (3) `ans[i] = ans[i >> 1] + (i & 1)`.

**Bài 4** — (1) `i & (i - 1)` bỏ 1 bit. (2) Cộng lại 1. (3) Vòng lặp `range(1, n + 1)`.

**Bài 5** — (1) `1` có 0 bit 0. (2) Số chẵn thêm một bit 0 ở cuối. (3) `zeros[0] = 1`, `zeros[1] = 0`, các số khác theo công thức.

**Bài 6** — (1) `key=lambda x: (popcount(x), x)`. (2) Tuple so sánh từng phần tử theo thứ tự. (3) Có thể dùng `bin(x).count("1")` làm popcount.

## 18. Đáp án tham khảo

**Bài 1:** `ans[9..15] = [2, 2, 3, 2, 3, 3, 4]`.

**Bài 2:** `6`, `1`, `8`, `'1010'`, `(3 + 4) & 1 = 7 & 1 = 1`.

In [29]:
def answer_count_bits(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans

def answer_count_bits_lowbit(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i & (i - 1)] + 1
    return ans

def answer_count_zero_bits(n: int) -> list[int]:
    zeros = [0] * (n + 1)
    zeros[0] = 1
    for i in range(2, n + 1):
        zeros[i] = zeros[i >> 1] + (1 - (i & 1))
    return zeros

def answer_sort_by_bits(arr: list[int]) -> list[int]:
    return sorted(arr, key=lambda x: (bin(x).count("1"), x))

assert answer_count_bits(15)[9:16] == [2, 2, 3, 2, 3, 3, 4]
assert (13 >> 1, 13 & 1, 12 & 11, bin(10)[2:], 3 + 4 & 1) == (6, 1, 8, "1010", 1)
assert answer_count_bits(5) == [0, 1, 1, 2, 1, 2] and answer_count_bits(0) == [0]
assert answer_count_bits_lowbit(8) == [0, 1, 1, 2, 1, 2, 2, 3, 1]
assert answer_count_zero_bits(8) == [1, 0, 1, 0, 2, 1, 1, 0, 3]
assert all(answer_count_zero_bits(300)[i] == bin(i)[2:].count("0") for i in range(301))
assert answer_sort_by_bits([0, 1, 2, 3, 4, 5, 6, 7, 8]) == [0, 1, 2, 4, 8, 3, 5, 6, 7]
assert answer_sort_by_bits([1024, 512, 256, 128, 64, 32, 16, 8, 4, 2, 1]) == [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countBits(self, n: int) -> list[int]:
        ans = [0] * (n + 1)
        for i in range(1, n + 1):
            ans[i] = ans[i >> 1] + (i & 1)     # hoặc ans[i & (i - 1)] + 1
        return ans
```

**Template DP trên chỉ số:** xác định `ans[0]` → tìm số **nhỏ hơn** `j = f(i)` có quan hệ đơn giản với `i` → `ans[i] = ans[j] + phần_chênh` → duyệt `i` tăng dần.

**Checklist trước khi submit:**
- [ ] Mảng dài `n + 1`, xử lý `n = 0` trả `[0]`?
- [ ] Có ngoặc quanh `(i & 1)`?
- [ ] Dùng `>>` (dịch phải), không phải `<<`?
- [ ] Duyệt `i` tăng dần để ô cần dùng đã được tính?
- [ ] Đã thử `n = 0, 1, 2, 5, 16`?